# Notebook-first application walkthrough

**Problem / objective:** Turn raw transaction history into defensible customer segments that marketing teams can actually act on.

**Decision / solution:** Prioritise segments using recency, frequency, monetary value, stability and commercial opportunity instead of relying on cluster labels alone.

This front section is intentionally analysis-first. It uses direct notebook code for inspection, EDA, visualisation and evidence review. The original notebook work is preserved below, followed by modular production code where that adds engineering evidence.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
PROJECT_SLUG = 'retail_customer_segmentation'
ROOT = Path.cwd()
if not (ROOT / 'projects').exists():
    candidate = ROOT.parent.parent if ROOT.name == PROJECT_SLUG else ROOT
    if (candidate / 'projects').exists():
        ROOT = candidate
PROJECT = ROOT / 'projects' / PROJECT_SLUG
if not PROJECT.exists() and Path.cwd().name == PROJECT_SLUG:
    PROJECT = Path.cwd()
    ROOT = PROJECT.parent.parent
assert PROJECT.exists(), f'Project directory not found: {PROJECT}'
print('Repository root:', ROOT.resolve())
print('Project:', PROJECT.resolve())


## 1. Find the real data and retained evidence

Instead of hiding the dataset behind a helper function, start by seeing what the project actually ships: raw/small data, fixtures, outputs, results and verified evidence. External large datasets remain reproducibly downloadable from the documented source.


In [ ]:
candidate_files = []
for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
    candidate_files.extend(PROJECT.rglob(pattern))
verified_dir = ROOT / 'verified' / PROJECT_SLUG
if verified_dir.exists():
    for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
        candidate_files.extend(verified_dir.rglob(pattern))
candidate_files = sorted({p.resolve() for p in candidate_files if p.is_file()})
file_inventory = pd.DataFrame({
    'file': [str(p.relative_to(ROOT)) if ROOT in p.parents else str(p) for p in candidate_files],
    'suffix': [p.suffix.lower() for p in candidate_files],
    'size_kb': [round(p.stat().st_size / 1024, 1) for p in candidate_files],
})
display(file_inventory.head(40))
print(f'Inspectable local data/evidence files: {len(file_inventory):,}')


## 2. Direct tabular data audit

The code below deliberately avoids a project-specific wrapper. It opens the first sensible local tabular asset, shows its schema and quality profile, and makes the data issues visible before modelling. If the full raw dataset is external, run the project's documented download cell/entry point first and rerun this section.


In [ ]:
tabular_candidates = [p for p in candidate_files if p.suffix.lower() in {'.csv', '.tsv', '.parquet'}]
preferred = [p for p in tabular_candidates if not any(token in p.name.lower() for token in ('metric', 'summary', 'verification'))]
tabular_path = (preferred or tabular_candidates or [None])[0]
df = None
if tabular_path is not None:
    if tabular_path.suffix.lower() == '.parquet':
        df = pd.read_parquet(tabular_path)
    else:
        sep = '\t' if tabular_path.suffix.lower() == '.tsv' else ','
        df = pd.read_csv(tabular_path, sep=sep, nrows=200_000)
    print('Loaded:', tabular_path)
    print('Shape:', df.shape)
    display(df.head())
    audit = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing': df.isna().sum(),
        'missing_pct': (100 * df.isna().mean()).round(2),
        'unique': df.nunique(dropna=False),
    }).sort_values(['missing_pct', 'unique'], ascending=[False, False])
    display(audit.head(30))
    print('Duplicate rows:', int(df.duplicated().sum()))
else:
    print('No local CSV/TSV/Parquet found yet. Use the project README/run path to download or build the documented dataset, then rerun this audit.')


## 3. Exploratory data analysis and visualisation

These plots are intentionally created in the notebook rather than described in prose. They expose distribution, missingness, scale, category balance and numeric relationships before any final model decision.


In [ ]:
if df is not None and len(df):
    missing_pct = (100 * df.isna().mean()).sort_values(ascending=False).head(20)
    missing_pct = missing_pct[missing_pct > 0]
    if len(missing_pct):
        plt.figure(figsize=(10, 4))
        missing_pct.plot(kind='bar')
        plt.title('Missing values by feature (%)')
        plt.ylabel('Missing %')
        plt.xticks(rotation=60, ha='right')
        plt.tight_layout()
        plt.show()

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:8]
    for col in numeric_cols:
        series = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(series):
            plt.figure(figsize=(8, 4))
            plt.hist(series, bins=30, alpha=0.8)
            plt.axvline(series.median(), linestyle='--', label=f'median={series.median():.2f}')
            plt.title(f'Distribution: {col}')
            plt.xlabel(col)
            plt.ylabel('Count')
            plt.legend()
            plt.tight_layout()
            plt.show()

    categorical_cols = [c for c in df.columns if c not in numeric_cols and df[c].nunique(dropna=False) <= 30][:4]
    for col in categorical_cols:
        counts = df[col].fillna('<missing>').astype(str).value_counts().head(15)
        plt.figure(figsize=(9, 4))
        counts.sort_values().plot(kind='barh')
        plt.title(f'Top categories: {col}')
        plt.xlabel('Rows')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr(numeric_only=True)
        plt.figure(figsize=(8, 6))
        image = plt.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
        plt.colorbar(image, label='Correlation')
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=60, ha='right')
        plt.yticks(range(len(corr.index)), corr.index)
        plt.title('Numeric correlation matrix')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        x_col, y_col = numeric_cols[0], numeric_cols[-1]
        sample = df[[x_col, y_col]].dropna().sample(min(3000, len(df.dropna(subset=[x_col, y_col]))), random_state=42)
        if len(sample):
            plt.figure(figsize=(7, 5))
            plt.scatter(sample[x_col], sample[y_col], alpha=0.35, s=18)
            plt.xlabel(x_col)
            plt.ylabel(y_col)
            plt.title(f'{y_col} versus {x_col}')
            plt.tight_layout()
            plt.show()
else:
    print('Run the documented data-build/download path, then rerun this section to render raw-data EDA.')


## 4. Inspect the measured results, not just the code

A portfolio project is stronger when it retains evidence. This section reads machine-readable JSON/CSV outputs and turns scalar metrics into a quick visual comparison.


In [ ]:
json_files = [p for p in candidate_files if p.suffix.lower() == '.json']
metric_rows = []
for path in json_files[:30]:
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        continue
    stack = [('', payload)]
    while stack:
        prefix, value = stack.pop()
        if isinstance(value, dict):
            for key, child in value.items():
                stack.append((f'{prefix}.{key}' if prefix else str(key), child))
        elif isinstance(value, (int, float)) and not isinstance(value, bool) and np.isfinite(value):
            metric_rows.append({
                'file': str(path.relative_to(ROOT)) if ROOT in path.parents else str(path),
                'metric': prefix,
                'value': float(value),
            })
metrics_df = pd.DataFrame(metric_rows)
if len(metrics_df):
    display(metrics_df.head(40))
    plot_df = metrics_df[np.isfinite(metrics_df['value'])].copy()
    plot_df = plot_df[plot_df['value'].abs() < 1_000_000].head(20)
    if len(plot_df):
        labels = (plot_df['file'].str.split('/').str[-1] + ' :: ' + plot_df['metric']).tolist()
        plt.figure(figsize=(10, max(4, 0.35 * len(plot_df))))
        plt.barh(range(len(plot_df)), plot_df['value'])
        plt.yticks(range(len(plot_df)), labels)
        plt.title('Retained project metrics / evidence')
        plt.tight_layout()
        plt.show()
else:
    print('No scalar JSON evidence found. Run the project and retain metrics/results before treating it as complete.')


## 5. Reproduce the application

The notebook should be understandable without running anything, but a reviewer can reproduce the canonical application below. The switch is off by default so opening the notebook never triggers a long training job unexpectedly.


In [ ]:
RUN_PROJECT = False
entrypoint = PROJECT / 'run.py'
if RUN_PROJECT and entrypoint.exists():
    subprocess.run([sys.executable, str(entrypoint)], cwd=PROJECT, check=True)
elif entrypoint.exists():
    print(f'Reproduce with: cd {PROJECT} && {sys.executable} run.py')
else:
    print('This project uses a different documented entry point; see README.md in the project folder.')


## 6. Decision / solution

Prioritise segments using recency, frequency, monetary value, stability and commercial opportunity instead of relying on cluster labels alone.

The final recommendation should be tied to the measured validation evidence and error analysis below. A model is not the solution by itself; the solution is the decision process built around it.


# Retail Customer Segmentation — Full Python Code

**Hiring purpose:** one project, one notebook, with the actual Python implementation visible. The modular files remain in the repository because that is how production code should be organised; this notebook mirrors those files so a recruiter can inspect the full code without hunting.


## Dataset and reproducibility

UCI Online Retail; raw workbook is downloaded reproducibly by src/data.py.

The project README/data card documents provenance, constraints and the exact reproduction path. Large third-party raw files are not duplicated in Git when licensing or repository size makes that poor engineering practice.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

PROJECT_SLUG = 'retail_customer_segmentation'
ROOT = Path.cwd()
if not (ROOT / 'projects').exists():
    target = Path('/content/uni_projects')
    if not target.exists():
        subprocess.run(['git', 'clone', 'https://github.com/Jorgoluka100/uni_projects.git', str(target)], check=True)
    os.chdir(target)
    ROOT = target
PROJECT = ROOT / 'projects' / PROJECT_SLUG
assert PROJECT.exists(), PROJECT
print('Project:', PROJECT.resolve())


## Full Python implementation

Every code cell below is copied directly from the corresponding `.py` file on the same commit. These cells are intentionally tagged `source-mirror` so the notebook acts as a readable code portfolio while the canonical modules remain testable files.


### `run.py`


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

from src.data import audit_raw_data, clean_transactions, download_dataset, load_raw_transactions, save_json
from src.evaluation import add_relative_segment_labels, summarize_clusters, validate_segment_output
from src.features import build_customer_features, prepare_clustering_matrix
from src.model import cluster_stability, evaluate_cluster_counts, fit_final_kmeans

PROJECT_DIR = Path(__file__).resolve().parent
DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"


def main() -> None:
    raw_dir = DATA_DIR / "raw"
    processed_dir = DATA_DIR / "processed"
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    processed_dir.mkdir(parents=True, exist_ok=True)

    workbook = download_dataset(raw_dir)
    raw = load_raw_transactions(workbook)
    raw_audit = audit_raw_data(raw)

    clean, cleaning_report = clean_transactions(raw)
    customers = build_customer_features(clean)
    matrix, transformed, _, preprocessing_metadata = prepare_clustering_matrix(customers)

    selection = evaluate_cluster_counts(matrix)
    model, labels = fit_final_kmeans(matrix, selection.selected_k)
    validate_segment_output(customers, labels)

    stability = cluster_stability(matrix, selection.selected_k, labels)
    summary = add_relative_segment_labels(summarize_clusters(customers, labels))

    assignments = customers.copy()
    assignments["cluster"] = labels

    clean.to_csv(processed_dir / "clean_transactions.csv", index=False)
    assignments.to_csv(RESULTS_DIR / "customer_segments.csv", index=False)
    selection.diagnostics.to_csv(RESULTS_DIR / "cluster_diagnostics.csv", index=False)
    summary.to_csv(RESULTS_DIR / "cluster_summary.csv", index=False)

    verification = {
        "verification_pass": True,
        "source": "UCI Machine Learning Repository - Online Retail",
        "raw_audit": raw_audit,
        "cleaning": cleaning_report,
        "customer_feature_rows": int(len(customers)),
        "selected_k": int(selection.selected_k),
        "selected_silhouette": float(
            selection.diagnostics.loc[
                selection.diagnostics["k"].eq(selection.selected_k), "silhouette"
            ].iloc[0]
        ),
        "cluster_stability": stability,
        "preprocessing": preprocessing_metadata,
        "cluster_centres_scaled": model.cluster_centers_.tolist(),
        "output_files": [
            "results/customer_segments.csv",
            "results/cluster_diagnostics.csv",
            "results/cluster_summary.csv",
        ],
        "limitations": [
            "KMeans imposes distance-based partitions and does not prove that natural customer segments exist.",
            "RFM summarizes transaction behaviour and does not capture demographics, channel exposure or profit margin.",
            "Segment names are relative descriptions of this dataset, not universal customer personas.",
            "The analysis is descriptive and should not be interpreted as causal evidence for marketing actions.",
        ],
    }
    save_json(verification, RESULTS_DIR / "verification.json")
    save_json(raw_audit, RESULTS_DIR / "raw_data_audit.json")
    save_json(cleaning_report, RESULTS_DIR / "cleaning_report.json")

    print(json.dumps(verification, indent=2))


if __name__ == "__main__":
    main()


### `src/data.py`


In [ ]:
from __future__ import annotations

import json
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

DATASET_URL = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"
EXPECTED_COLUMNS = {
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country",
}


def download_dataset(cache_dir: Path) -> Path:
    """Download the UCI Online Retail workbook and return the local xlsx path."""
    cache_dir.mkdir(parents=True, exist_ok=True)
    archive_path = cache_dir / "online_retail.zip"

    if not archive_path.exists():
        request = urllib.request.Request(DATASET_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request, timeout=120) as response, archive_path.open("wb") as target:
            for chunk in iter(lambda: response.read(1 << 20), b""):
                target.write(chunk)

    with zipfile.ZipFile(archive_path) as archive:
        workbook_names = [name for name in archive.namelist() if name.lower().endswith(".xlsx")]
        if len(workbook_names) != 1:
            raise ValueError(f"Expected one xlsx workbook, found {workbook_names}")
        workbook_name = workbook_names[0]
        workbook_path = cache_dir / Path(workbook_name).name
        if not workbook_path.exists():
            archive.extract(workbook_name, cache_dir)
            extracted = cache_dir / workbook_name
            if extracted != workbook_path:
                extracted.replace(workbook_path)
    return workbook_path


def load_raw_transactions(workbook_path: Path) -> pd.DataFrame:
    frame = pd.read_excel(workbook_path, engine="openpyxl")
    missing = sorted(EXPECTED_COLUMNS - set(frame.columns))
    if missing:
        raise ValueError(f"Source schema is missing expected columns: {missing}")
    return frame


def audit_raw_data(frame: pd.DataFrame) -> dict:
    """Return a machine-readable profile before cleaning."""
    return {
        "rows": int(len(frame)),
        "columns": int(frame.shape[1]),
        "exact_duplicate_rows": int(frame.duplicated().sum()),
        "missing_by_column": {key: int(value) for key, value in frame.isna().sum().items()},
        "non_positive_quantity_rows": int((pd.to_numeric(frame["Quantity"], errors="coerce") <= 0).sum()),
        "non_positive_unit_price_rows": int((pd.to_numeric(frame["UnitPrice"], errors="coerce") <= 0).sum()),
        "cancelled_invoice_rows": int(frame["InvoiceNo"].astype("string").str.upper().str.startswith("C", na=False).sum()),
        "unique_invoices": int(frame["InvoiceNo"].nunique(dropna=True)),
        "unique_customers": int(frame["CustomerID"].nunique(dropna=True)),
        "date_min": str(pd.to_datetime(frame["InvoiceDate"], errors="coerce").min()),
        "date_max": str(pd.to_datetime(frame["InvoiceDate"], errors="coerce").max()),
    }


def clean_transactions(frame: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Create a customer-level modelling base from messy transactional data.

    Rules are explicit and auditable rather than silently dropping rows:
    - remove exact duplicate lines;
    - require customer, invoice, stock code, timestamp and country;
    - remove cancellations and non-positive quantity/price rows;
    - normalize text and dtypes;
    - create line revenue and validate the final table.
    """
    work = frame.copy()
    initial_rows = len(work)

    duplicate_mask = work.duplicated(keep="first")
    duplicate_rows = int(duplicate_mask.sum())
    work = work.loc[~duplicate_mask].copy()

    for column in ["InvoiceNo", "StockCode", "Description", "Country"]:
        work[column] = work[column].astype("string").str.strip()

    work["InvoiceDate"] = pd.to_datetime(work["InvoiceDate"], errors="coerce")
    work["Quantity"] = pd.to_numeric(work["Quantity"], errors="coerce")
    work["UnitPrice"] = pd.to_numeric(work["UnitPrice"], errors="coerce")
    work["CustomerID"] = pd.to_numeric(work["CustomerID"], errors="coerce")

    work["Country"] = work["Country"].str.replace(r"\s+", " ", regex=True)
    work["Description"] = work["Description"].str.replace(r"\s+", " ", regex=True)
    work["is_cancelled"] = work["InvoiceNo"].str.upper().str.startswith("C", na=False)

    reason_masks = {
        "missing_customer_id": work["CustomerID"].isna(),
        "missing_invoice_no": work["InvoiceNo"].isna() | work["InvoiceNo"].eq(""),
        "missing_stock_code": work["StockCode"].isna() | work["StockCode"].eq(""),
        "missing_invoice_date": work["InvoiceDate"].isna(),
        "missing_country": work["Country"].isna() | work["Country"].eq(""),
        "cancelled_invoice": work["is_cancelled"],
        "non_positive_quantity": work["Quantity"].isna() | work["Quantity"].le(0),
        "non_positive_unit_price": work["UnitPrice"].isna() | work["UnitPrice"].le(0),
    }

    invalid_mask = np.zeros(len(work), dtype=bool)
    removed_by_rule: dict[str, int] = {}
    for name, mask in reason_masks.items():
        removed_by_rule[name] = int(mask.sum())
        invalid_mask |= mask.to_numpy()

    clean = work.loc[~invalid_mask].copy()
    clean["CustomerID"] = clean["CustomerID"].round().astype("int64").astype("string")
    clean["Quantity"] = clean["Quantity"].astype("int64")
    clean["line_revenue"] = clean["Quantity"] * clean["UnitPrice"]

    clean = clean[
        [
            "InvoiceNo",
            "InvoiceDate",
            "CustomerID",
            "StockCode",
            "Description",
            "Quantity",
            "UnitPrice",
            "line_revenue",
            "Country",
        ]
    ].sort_values(["InvoiceDate", "InvoiceNo", "StockCode"], kind="stable")
    clean = clean.reset_index(drop=True)

    if clean.empty:
        raise ValueError("Cleaning removed every row; inspect source or cleaning rules")
    if clean.duplicated().any():
        raise AssertionError("Exact duplicates remain after cleaning")
    if clean[["InvoiceNo", "InvoiceDate", "CustomerID", "StockCode", "Quantity", "UnitPrice", "Country"]].isna().any().any():
        raise AssertionError("Required final fields contain missing values")
    if not clean["Quantity"].gt(0).all():
        raise AssertionError("Final dataset contains non-positive quantities")
    if not clean["UnitPrice"].gt(0).all():
        raise AssertionError("Final dataset contains non-positive prices")
    if not clean["line_revenue"].gt(0).all():
        raise AssertionError("Final dataset contains non-positive line revenue")

    report = {
        "source_rows": int(initial_rows),
        "exact_duplicates_removed": duplicate_rows,
        "rule_counts_before_combining": removed_by_rule,
        "rows_removed_after_deduplication": int(len(work) - len(clean)),
        "final_rows": int(len(clean)),
        "retained_share": float(len(clean) / initial_rows),
        "final_unique_customers": int(clean["CustomerID"].nunique()),
        "final_unique_invoices": int(clean["InvoiceNo"].nunique()),
        "final_revenue": float(clean["line_revenue"].sum()),
        "final_date_min": clean["InvoiceDate"].min().isoformat(),
        "final_date_max": clean["InvoiceDate"].max().isoformat(),
    }
    return clean, report


def save_json(payload: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")


### `src/features.py`


In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler

RFM_COLUMNS = ["recency_days", "frequency_orders", "monetary_value"]


def build_customer_features(transactions: pd.DataFrame) -> pd.DataFrame:
    """Aggregate clean line-level transactions into one row per customer."""
    required = {
        "CustomerID",
        "InvoiceNo",
        "InvoiceDate",
        "Quantity",
        "line_revenue",
    }
    missing = sorted(required - set(transactions.columns))
    if missing:
        raise ValueError(f"Missing required transaction columns: {missing}")

    snapshot_date = transactions["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)

    customer = (
        transactions.groupby("CustomerID", as_index=False)
        .agg(
            last_purchase=("InvoiceDate", "max"),
            first_purchase=("InvoiceDate", "min"),
            frequency_orders=("InvoiceNo", "nunique"),
            monetary_value=("line_revenue", "sum"),
            total_items=("Quantity", "sum"),
            transaction_lines=("InvoiceNo", "size"),
        )
    )

    customer["recency_days"] = (snapshot_date - customer["last_purchase"].dt.normalize()).dt.days
    customer["customer_tenure_days"] = (
        customer["last_purchase"].dt.normalize() - customer["first_purchase"].dt.normalize()
    ).dt.days + 1
    customer["average_order_value"] = customer["monetary_value"] / customer["frequency_orders"]

    if customer["CustomerID"].duplicated().any():
        raise AssertionError("Customer feature table is not one row per customer")
    if customer[RFM_COLUMNS].isna().any().any():
        raise AssertionError("RFM features contain missing values")
    if customer["recency_days"].lt(0).any():
        raise AssertionError("Recency cannot be negative")
    if customer["frequency_orders"].le(0).any():
        raise AssertionError("Frequency must be positive")
    if customer["monetary_value"].le(0).any():
        raise AssertionError("Monetary value must be positive")

    return customer.sort_values("CustomerID").reset_index(drop=True)


def cap_extreme_values(
    frame: pd.DataFrame,
    columns: list[str] | None = None,
    lower_quantile: float = 0.01,
    upper_quantile: float = 0.99,
) -> tuple[pd.DataFrame, dict]:
    """Winsorise extreme RFM values while keeping every customer in the analysis."""
    columns = columns or RFM_COLUMNS
    capped = frame.copy()
    caps: dict[str, dict[str, float]] = {}

    for column in columns:
        lower = float(capped[column].quantile(lower_quantile))
        upper = float(capped[column].quantile(upper_quantile))
        if lower > upper:
            raise ValueError(f"Invalid quantile bounds for {column}")
        capped[column] = capped[column].clip(lower=lower, upper=upper)
        caps[column] = {"lower": lower, "upper": upper}

    return capped, caps


def prepare_clustering_matrix(
    customer_features: pd.DataFrame,
) -> tuple[np.ndarray, pd.DataFrame, RobustScaler, dict]:
    """Log-transform skewed RFM features and robust-scale them for distance models."""
    capped, caps = cap_extreme_values(customer_features)

    transformed = pd.DataFrame(index=capped.index)
    transformed["recency_log1p"] = np.log1p(capped["recency_days"].astype(float))
    transformed["frequency_log1p"] = np.log1p(capped["frequency_orders"].astype(float))
    transformed["monetary_log1p"] = np.log1p(capped["monetary_value"].astype(float))

    scaler = RobustScaler()
    matrix = scaler.fit_transform(transformed)

    if not np.isfinite(matrix).all():
        raise AssertionError("Clustering matrix contains non-finite values")

    metadata = {
        "winsorisation_caps": caps,
        "transforms": {
            "recency_days": "log1p after 1st/99th percentile clipping",
            "frequency_orders": "log1p after 1st/99th percentile clipping",
            "monetary_value": "log1p after 1st/99th percentile clipping",
        },
        "scaler": "RobustScaler",
    }
    return matrix, transformed, scaler, metadata


### `src/model.py`


In [ ]:
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)

SEED = 42


@dataclass(frozen=True)
class ClusterSelection:
    selected_k: int
    diagnostics: pd.DataFrame


def evaluate_cluster_counts(
    matrix: np.ndarray,
    k_values: range | list[int] = range(2, 11),
    seed: int = SEED,
) -> ClusterSelection:
    """Compare candidate k values with multiple complementary diagnostics."""
    if len(matrix) < 10:
        raise ValueError("Too few customers for meaningful clustering")

    rows: list[dict] = []
    for k in k_values:
        if k >= len(matrix):
            continue
        model = KMeans(n_clusters=k, n_init=50, random_state=seed)
        labels = model.fit_predict(matrix)
        rows.append(
            {
                "k": int(k),
                "silhouette": float(silhouette_score(matrix, labels)),
                "davies_bouldin": float(davies_bouldin_score(matrix, labels)),
                "calinski_harabasz": float(calinski_harabasz_score(matrix, labels)),
                "inertia": float(model.inertia_),
                "smallest_cluster_share": float(pd.Series(labels).value_counts(normalize=True).min()),
            }
        )

    diagnostics = pd.DataFrame(rows).sort_values("k").reset_index(drop=True)
    if diagnostics.empty:
        raise ValueError("No valid cluster counts were evaluated")

    # Silhouette is the primary selection rule. A minimum cluster-size guard avoids
    # promoting a superficially strong solution dominated by tiny fragments.
    eligible = diagnostics.loc[diagnostics["smallest_cluster_share"] >= 0.02]
    selection_pool = eligible if not eligible.empty else diagnostics
    selected_k = int(selection_pool.sort_values(["silhouette", "k"], ascending=[False, True]).iloc[0]["k"])
    return ClusterSelection(selected_k=selected_k, diagnostics=diagnostics)


def fit_final_kmeans(matrix: np.ndarray, n_clusters: int, seed: int = SEED) -> tuple[KMeans, np.ndarray]:
    model = KMeans(n_clusters=n_clusters, n_init=100, random_state=seed)
    labels = model.fit_predict(matrix)
    return model, labels


def cluster_stability(
    matrix: np.ndarray,
    n_clusters: int,
    reference_labels: np.ndarray,
    seeds: tuple[int, ...] = (7, 19, 31, 43, 59, 71, 83, 97),
) -> dict:
    """Estimate sensitivity to KMeans initialization using adjusted Rand index."""
    scores = []
    for seed in seeds:
        model = KMeans(n_clusters=n_clusters, n_init=20, random_state=seed)
        candidate = model.fit_predict(matrix)
        scores.append(float(adjusted_rand_score(reference_labels, candidate)))

    return {
        "metric": "adjusted_rand_index",
        "runs": len(scores),
        "mean": float(np.mean(scores)),
        "min": float(np.min(scores)),
        "max": float(np.max(scores)),
        "scores": scores,
    }


### `src/evaluation.py`


In [ ]:
from __future__ import annotations

import pandas as pd


def summarize_clusters(customer_features: pd.DataFrame, labels) -> pd.DataFrame:
    """Return an interpretable customer-segment table using original business units."""
    segmented = customer_features.copy()
    segmented["cluster"] = labels

    summary = (
        segmented.groupby("cluster", as_index=False)
        .agg(
            customers=("CustomerID", "size"),
            median_recency_days=("recency_days", "median"),
            median_frequency_orders=("frequency_orders", "median"),
            median_monetary_value=("monetary_value", "median"),
            mean_order_value=("average_order_value", "mean"),
            total_revenue=("monetary_value", "sum"),
        )
        .sort_values("median_monetary_value", ascending=False)
        .reset_index(drop=True)
    )
    summary["customer_share"] = summary["customers"] / summary["customers"].sum()
    summary["revenue_share"] = summary["total_revenue"] / summary["total_revenue"].sum()
    return summary


def add_relative_segment_labels(summary: pd.DataFrame) -> pd.DataFrame:
    """Attach descriptive labels based on observed cluster profiles, not hidden assumptions."""
    labelled = summary.copy()
    recency_mid = labelled["median_recency_days"].median()
    frequency_mid = labelled["median_frequency_orders"].median()
    monetary_mid = labelled["median_monetary_value"].median()

    def describe(row) -> str:
        recent = row["median_recency_days"] <= recency_mid
        frequent = row["median_frequency_orders"] >= frequency_mid
        valuable = row["median_monetary_value"] >= monetary_mid
        if recent and frequent and valuable:
            return "high_value_active"
        if recent and not frequent and valuable:
            return "recent_high_spend"
        if recent and not valuable:
            return "recent_lower_value"
        if not recent and valuable:
            return "valuable_at_risk"
        return "inactive_lower_value"

    labelled["relative_profile"] = labelled.apply(describe, axis=1)
    return labelled


def validate_segment_output(customer_features: pd.DataFrame, labels) -> None:
    if len(customer_features) != len(labels):
        raise AssertionError("Every customer must receive exactly one cluster label")
    if pd.Series(labels).isna().any():
        raise AssertionError("Cluster labels contain missing values")
    if pd.Series(labels).nunique() < 2:
        raise AssertionError("Clustering collapsed to fewer than two segments")


## Run the real project

The cell below executes the canonical project entry point rather than a rewritten toy version. Keep `RUN_PIPELINE = False` when you only want to inspect the notebook; change it to `True` to reproduce the project.


In [ ]:
RUN_PIPELINE = False
if RUN_PIPELINE:
    subprocess.run([sys.executable, 'run.py'], cwd=PROJECT, check=True)
else:
    print(f'Reproduce with: cd {PROJECT} && python run.py')


In [ ]:
evidence = []
for folder in (PROJECT / 'results', ROOT / 'verified' / PROJECT_SLUG):
    if folder.exists():
        evidence.extend(sorted(folder.glob('*.json')))
for path in evidence[:5]:
    print('\n---', path.relative_to(ROOT), '---')
    print(path.read_text(encoding='utf-8')[:12000])


## Interview discussion

Be ready to explain the business problem, dataset provenance, cleaning/preprocessing decisions, leakage controls, modelling or analytical choices, evaluation design, limitations, testing strategy and what you would change in production. The key signal is that the notebook, modular source, tests and retained evidence all tell the same story.


# Deeper exploratory analysis and retained evidence

These direct notebook cells extend the initial EDA with data-quality, scale, relationship, output and error diagnostics. They are intentionally visible here rather than hidden behind project helper functions.


In [ ]:
# Extended data-quality scorecard
if df is not None and len(df):
    quality_rows = []
    for col in df.columns:
        series = df[col]
        row = {
            'feature': col,
            'dtype': str(series.dtype),
            'rows': len(series),
            'missing': int(series.isna().sum()),
            'missing_pct': float(100 * series.isna().mean()),
            'unique': int(series.nunique(dropna=False)),
            'unique_pct': float(100 * series.nunique(dropna=False) / max(len(series), 1)),
        }
        if pd.api.types.is_numeric_dtype(series):
            values = pd.to_numeric(series, errors='coerce').dropna()
            if len(values):
                q1, q3 = values.quantile([0.25, 0.75])
                iqr = q3 - q1
                row.update({
                    'mean': float(values.mean()),
                    'median': float(values.median()),
                    'std': float(values.std()),
                    'p05': float(values.quantile(0.05)),
                    'p95': float(values.quantile(0.95)),
                    'skew': float(values.skew()),
                    'iqr_outliers': int(((values < q1 - 1.5*iqr) | (values > q3 + 1.5*iqr)).sum()),
                })
        quality_rows.append(row)
    deep_quality = pd.DataFrame(quality_rows)
    display(deep_quality.sort_values(['missing_pct','unique'], ascending=[False,False]).head(40))
    if 'iqr_outliers' in deep_quality:
        outlier_view = deep_quality.dropna(subset=['iqr_outliers']).sort_values('iqr_outliers', ascending=False).head(15)
        if len(outlier_view):
            plt.figure(figsize=(10,4))
            plt.bar(outlier_view['feature'], outlier_view['iqr_outliers'])
            plt.title('Potential IQR outliers by feature')
            plt.ylabel('Rows')
            plt.xticks(rotation=60, ha='right')
            plt.tight_layout()
            plt.show()
    card = deep_quality.sort_values('unique', ascending=False).head(20)
    plt.figure(figsize=(10,4))
    plt.bar(card['feature'], card['unique'])
    plt.title('Feature cardinality')
    plt.ylabel('Unique values')
    plt.xticks(rotation=60, ha='right')
    plt.tight_layout()
    plt.show()
    print('Constant columns:', deep_quality.loc[deep_quality['unique'] <= 1, 'feature'].tolist())
    print('High-missing columns:', deep_quality.loc[deep_quality['missing_pct'] >= 30, 'feature'].tolist())
    print('Possible identifier columns:', deep_quality.loc[deep_quality['unique_pct'] >= 95, 'feature'].tolist()[:20])
else:
    print('Materialise the documented dataset to run the extended data-quality scorecard.')


In [ ]:
# Numeric distributions, spread and strongest pairwise relationships
if df is not None and len(df):
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:12]
    for col in numeric_cols:
        values = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(values) < 5:
            continue
        clipped = values.clip(values.quantile(0.01), values.quantile(0.99))
        plt.figure(figsize=(8,4))
        plt.hist(clipped, bins=35, alpha=0.82)
        plt.axvline(values.median(), linestyle='--', label=f'median={values.median():.3g}')
        plt.axvline(values.mean(), linestyle=':', label=f'mean={values.mean():.3g}')
        plt.title(f'Distribution: {col} (1st–99th percentile)')
        plt.xlabel(col)
        plt.ylabel('Rows')
        plt.legend()
        plt.tight_layout()
        plt.show()
        plt.figure(figsize=(8,3))
        plt.boxplot(values, vert=False, showfliers=True)
        plt.title(f'Spread / outliers: {col}')
        plt.xlabel(col)
        plt.tight_layout()
        plt.show()
    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr(numeric_only=True)
        pairs = []
        for i, left in enumerate(corr.columns):
            for right in corr.columns[i+1:]:
                value = corr.loc[left, right]
                if pd.notna(value):
                    pairs.append({'feature_a': left, 'feature_b': right, 'correlation': float(value), 'abs_correlation': float(abs(value))})
        corr_pairs = pd.DataFrame(pairs).sort_values('abs_correlation', ascending=False) if pairs else pd.DataFrame()
        if len(corr_pairs):
            display(corr_pairs.head(20).round(4))
            for _, pair in corr_pairs.head(4).iterrows():
                sample = df[[pair['feature_a'], pair['feature_b']]].dropna()
                if len(sample) > 3000:
                    sample = sample.sample(3000, random_state=42)
                plt.figure(figsize=(7,5))
                plt.scatter(sample[pair['feature_a']], sample[pair['feature_b']], alpha=0.30, s=16)
                plt.xlabel(pair['feature_a'])
                plt.ylabel(pair['feature_b'])
                plt.title(f"{pair['feature_a']} vs {pair['feature_b']} (r={pair['correlation']:.2f})")
                plt.tight_layout()
                plt.show()
    categorical = [c for c in df.columns if 2 <= df[c].nunique(dropna=False) <= 20][:8]
    for col in categorical:
        counts = df[col].fillna('<missing>').astype(str).value_counts().head(20)
        shares = 100 * counts / counts.sum()
        display(pd.DataFrame({'rows': counts, 'share_pct': shares.round(2)}))
        plt.figure(figsize=(8,4))
        counts.sort_values().plot(kind='barh')
        plt.title(f'Category balance: {col}')
        plt.xlabel('Rows')
        plt.tight_layout()
        plt.show()
else:
    print('Materialise the documented dataset to run distribution diagnostics.')


In [ ]:
# Temporal coverage where date/time fields exist
if df is not None and len(df):
    time_cols = [c for c in df.columns if any(token in str(c).lower() for token in ('date','time','timestamp','datetime'))]
    print('Date/time candidates:', time_cols[:10])
    for col in time_cols[:4]:
        converted = pd.to_datetime(df[col], errors='coerce')
        valid = converted.dropna()
        if len(valid) >= max(10, int(0.25*len(df))):
            print(col, 'range:', valid.min(), '→', valid.max())
            monthly = valid.dt.to_period('M').value_counts().sort_index()
            if len(monthly) > 1:
                plt.figure(figsize=(10,4))
                plt.plot(monthly.index.astype(str), monthly.values, marker='o')
                plt.title(f'Rows over time: {col}')
                plt.ylabel('Rows')
                plt.xticks(rotation=70, ha='right')
                plt.tight_layout()
                plt.show()


## Retained outputs and error analysis

A strong portfolio keeps inspectable evidence. The cells below profile compact result tables and automatically detect prediction-like columns for residual or misclassification analysis.


In [ ]:
# Load compact result/evidence tables
result_tables = []
for base in [PROJECT/'results', PROJECT/'outputs', PROJECT/'artifacts', ROOT/'verified'/PROJECT_SLUG]:
    if not base.exists():
        continue
    for path in sorted(base.rglob('*')):
        if path.is_file() and path.suffix.lower() in {'.csv','.tsv','.parquet'} and path.stat().st_size < 25_000_000:
            try:
                if path.suffix.lower() == '.parquet':
                    table = pd.read_parquet(path)
                else:
                    table = pd.read_csv(path, sep='	' if path.suffix.lower() == '.tsv' else ',')
            except Exception as exc:
                print('Could not read', path.name, '-', exc)
                continue
            result_tables.append((path, table))
            print('
RESULT TABLE:', path.relative_to(ROOT) if ROOT in path.parents else path)
            print('shape=', table.shape)
            display(table.head(15))
            numeric = table.select_dtypes(include=np.number).columns.tolist()[:12]
            if numeric:
                display(table[numeric].describe().T.round(4))
print('Inspectable result tables:', len(result_tables))


In [ ]:
# Automatic regression/classification-style error diagnostics
actual_tokens = ('actual','target','truth','y_true','observed','label')
pred_tokens = ('prediction','predicted','forecast','y_pred')
confidence_tokens = ('confidence','probability','proba','risk','uncertainty')
for path, table in result_tables:
    actual_cols = [c for c in table.columns if any(token in str(c).lower() for token in actual_tokens)]
    pred_cols = [c for c in table.columns if any(token in str(c).lower() for token in pred_tokens)]
    conf_cols = [c for c in table.columns if any(token in str(c).lower() for token in confidence_tokens)]
    if actual_cols and pred_cols and len(table):
        actual_col = actual_cols[0]
        pred_col = next((c for c in pred_cols if c != actual_col), pred_cols[0])
        actual_num = pd.to_numeric(table[actual_col], errors='coerce')
        pred_num = pd.to_numeric(table[pred_col], errors='coerce')
        numeric_mask = actual_num.notna() & pred_num.notna()
        if numeric_mask.sum() >= 10:
            residual = actual_num[numeric_mask] - pred_num[numeric_mask]
            abs_error = residual.abs()
            print('
', path.name, '| MAE=', round(float(abs_error.mean()),5), '| RMSE=', round(float(np.sqrt(np.mean(residual**2))),5), '| bias=', round(float(residual.mean()),5))
            plt.figure(figsize=(7,5))
            plt.scatter(actual_num[numeric_mask], pred_num[numeric_mask], alpha=0.35, s=18)
            lo = min(actual_num[numeric_mask].min(), pred_num[numeric_mask].min())
            hi = max(actual_num[numeric_mask].max(), pred_num[numeric_mask].max())
            plt.plot([lo,hi],[lo,hi], linestyle='--')
            plt.xlabel(str(actual_col))
            plt.ylabel(str(pred_col))
            plt.title(f'Actual vs predicted — {path.name}')
            plt.tight_layout()
            plt.show()
            plt.figure(figsize=(7,4))
            plt.hist(residual, bins=30, alpha=0.82)
            plt.axvline(0, linestyle='--')
            plt.title(f'Residual distribution — {path.name}')
            plt.tight_layout()
            plt.show()
            worst_idx = abs_error.nlargest(min(15,len(abs_error))).index
            cols = list(dict.fromkeys([actual_col,pred_col]+conf_cols[:2]))
            worst = table.loc[worst_idx, cols].copy()
            worst['absolute_error'] = abs_error.loc[worst_idx].values
            display(worst.sort_values('absolute_error', ascending=False))
        else:
            agreement = table[actual_col].astype(str) == table[pred_col].astype(str)
            print('
', path.name, '| classification agreement=', round(float(agreement.mean()),4))
            if (~agreement).any():
                display(table.loc[~agreement, [actual_col,pred_col]+conf_cols[:2]].head(20))
    elif conf_cols:
        for col in conf_cols[:2]:
            values = pd.to_numeric(table[col], errors='coerce').dropna()
            if len(values) >= 10:
                plt.figure(figsize=(7,4))
                plt.hist(values, bins=30, alpha=0.82)
                plt.title(f'{col} distribution — {path.name}')
                plt.tight_layout()
                plt.show()


In [ ]:
# Display retained visual evidence from actual project runs
png_files = []
for base in [PROJECT/'results', PROJECT/'outputs', PROJECT/'artifacts', ROOT/'verified'/PROJECT_SLUG]:
    if base.exists():
        png_files.extend(sorted(base.rglob('*.png')))
print('Retained PNG figures:', len(png_files))
for path in png_files[:12]:
    try:
        image = plt.imread(path)
        plt.figure(figsize=(10,6))
        plt.imshow(image)
        plt.axis('off')
        plt.title(str(path.relative_to(ROOT)) if ROOT in path.parents else path.name)
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print('Could not display', path.name, '-', exc)


In [ ]:
# Reproducibility and evidence checklist
checks = [
    {'check':'README present', 'status':(PROJECT/'README.md').exists()},
    {'check':'Recruiter notebook present', 'status':(PROJECT/'project_notebook.ipynb').exists()},
    {'check':'Python implementation present', 'status':any(PROJECT.rglob('*.py'))},
    {'check':'Tests present', 'status':(PROJECT/'tests').exists() and any((PROJECT/'tests').rglob('test*.py'))},
    {'check':'Result/evidence files present', 'status':bool(candidate_files)},
    {'check':'Machine-readable JSON evidence', 'status':bool(json_files)},
    {'check':'Retained visual evidence', 'status':bool(png_files)},
]
checklist = pd.DataFrame(checks)
display(checklist)
print('Evidence checklist pass rate:', f"{100*checklist['status'].mean():.1f}%")
print('A failed item is a prompt to strengthen the project, not something to hide.')


# Engineering appendix — canonical application source

The analysis and visual evidence come first. The cells below preserve additional canonical Python from this project for reviewers who want to inspect pipelines, APIs, tests, feature code, monitoring and reusable implementation details.


## Canonical source: `src/__init__.py`


In [ ]:
"""Retail customer data-cleaning and segmentation project."""


## Canonical source: `tests/test_cleaning.py`


In [ ]:
import pandas as pd

from src.data import audit_raw_data, clean_transactions


def _fixture() -> pd.DataFrame:
    return pd.DataFrame(
        {
            "InvoiceNo": ["10001", "10001", "C10002", "10003", "10004", "10005", "10006"],
            "StockCode": ["A1", "A1", "A2", "A3", "A4", "A5", "A6"],
            "Description": [" Mug ", " Mug ", "Return", "Tea", "Plate", "Bowl", "Glass"],
            "Quantity": [2, 2, -1, 1, 0, 1, 3],
            "InvoiceDate": pd.to_datetime(
                [
                    "2025-01-01 09:00",
                    "2025-01-01 09:00",
                    "2025-01-02 10:00",
                    "2025-01-03 11:00",
                    "2025-01-04 12:00",
                    "2025-01-05 13:00",
                    "2025-01-06 14:00",
                ]
            ),
            "UnitPrice": [5.0, 5.0, 4.0, 3.0, 2.0, 0.0, 7.0],
            "CustomerID": [1, 1, 2, None, 4, 5, 6],
            "Country": [" United Kingdom "] * 7,
        }
    )


def test_raw_audit_counts_quality_problems():
    audit = audit_raw_data(_fixture())
    assert audit["rows"] == 7
    assert audit["exact_duplicate_rows"] == 1
    assert audit["cancelled_invoice_rows"] == 1
    assert audit["non_positive_quantity_rows"] == 2
    assert audit["non_positive_unit_price_rows"] == 1


def test_cleaning_is_explicit_and_produces_valid_rows():
    clean, report = clean_transactions(_fixture())

    assert report["exact_duplicates_removed"] == 1
    assert len(clean) == 2
    assert set(clean["CustomerID"]) == {"1", "6"}
    assert clean["Quantity"].gt(0).all()
    assert clean["UnitPrice"].gt(0).all()
    assert clean["line_revenue"].gt(0).all()
    assert not clean.duplicated().any()
    assert clean.loc[clean["CustomerID"].eq("1"), "Country"].iloc[0] == "United Kingdom"


## Canonical source: `tests/test_features_and_model.py`


In [ ]:
import numpy as np
import pandas as pd

from src.evaluation import summarize_clusters, validate_segment_output
from src.features import build_customer_features, prepare_clustering_matrix
from src.model import cluster_stability, evaluate_cluster_counts, fit_final_kmeans


def _clean_transactions() -> pd.DataFrame:
    rows = []
    start = pd.Timestamp("2025-01-01")
    for customer in range(1, 31):
        orders = 1 + (customer % 5)
        for order in range(orders):
            rows.append(
                {
                    "InvoiceNo": f"{customer:03d}-{order:02d}",
                    "InvoiceDate": start + pd.Timedelta(days=customer * 2 + order),
                    "CustomerID": str(customer),
                    "StockCode": f"S{order:02d}",
                    "Description": "fixture",
                    "Quantity": 1 + customer % 4,
                    "UnitPrice": 2.0 + customer,
                    "line_revenue": float((1 + customer % 4) * (2.0 + customer)),
                    "Country": "United Kingdom",
                }
            )
    return pd.DataFrame(rows)


def test_customer_features_are_one_row_per_customer():
    customer = build_customer_features(_clean_transactions())
    assert len(customer) == 30
    assert customer["CustomerID"].is_unique
    assert customer["recency_days"].ge(0).all()
    assert customer["frequency_orders"].gt(0).all()
    assert customer["monetary_value"].gt(0).all()


def test_clustering_pipeline_returns_stable_valid_shapes():
    customer = build_customer_features(_clean_transactions())
    matrix, transformed, _, metadata = prepare_clustering_matrix(customer)

    assert matrix.shape == (30, 3)
    assert transformed.shape == (30, 3)
    assert np.isfinite(matrix).all()
    assert metadata["scaler"] == "RobustScaler"

    selection = evaluate_cluster_counts(matrix, k_values=[2, 3, 4])
    assert selection.selected_k in {2, 3, 4}
    assert set(selection.diagnostics["k"]) == {2, 3, 4}

    _, labels = fit_final_kmeans(matrix, selection.selected_k)
    validate_segment_output(customer, labels)
    summary = summarize_clusters(customer, labels)
    assert summary["customers"].sum() == 30
    assert np.isclose(summary["customer_share"].sum(), 1.0)

    stability = cluster_stability(matrix, selection.selected_k, labels, seeds=(3, 5, 7))
    assert stability["runs"] == 3
    assert 0.0 <= stability["min"] <= 1.0


# Portfolio depth check

**Meaningful code lines visible in this notebook:** 694. For a major recruiter-facing application the working target is roughly **1,000 meaningful lines**, with a practical guide of about 600–1,400 depending on the problem. This notebook is within the major-project guide.

Line count is not a quality metric by itself. Add code only when it improves the real project: data acquisition, validation, cleaning, EDA, visualisation, feature engineering, baselines, model comparison, tuning, leakage control, error analysis, explainability, uncertainty, inference, tests, monitoring, deployment or decision logic.
